# Numerical Methods Week, Notebook 1: Estimation

Every method in this notebook exists for the same reason: **the exact answer is either
unavailable or too expensive, so we settle for a controlled approximation.**

That word *controlled* is the whole subject. Anyone can produce a wrong number. A
numerical method comes with a story about *how* wrong it is and *how fast* the error
shrinks as you work harder. Chasing that story is most of what you will do here.

Four exercises, each built in small steps:

1. **Divided differences**: the bookkeeping device behind interpolation, and a discrete
   stand-in for the derivative.
2. **Numerical integration**: approximating $\int_a^b f(x)\,dx$ from samples.
3. **Numerical differentiation**: approximating $f'(x)$ from samples, and the surprising
   way it fights back.
4. **Newton's method**: solving $f(x) = 0$ when algebra cannot.

## How to use this notebook

Each exercise is a sequence of small steps: write a little, **test it immediately**, then
extend. Do not write a whole function and hope. Numerical code fails quietly (it returns
a plausible-looking number that is simply wrong), so the habit of testing every piece
against a case you can verify by hand is not optional here, it is the job.

Run the cell below first; every exercise assumes these imports.

In [ ]:
# Setup: run this first.
# Import numpy as np and matplotlib.pyplot as plt.
# Then print numpy's version, to confirm the environment is working.

---
# Exercise 1: Divided differences

## The idea

Given $n+1$ distinct nodes $x_0, \dots, x_n$ and values $f(x_i)$, the **divided
differences** are defined recursively:

$$f[x_i] = f(x_i)$$

$$f[x_i, \dots, x_{i+k}] \;=\; \frac{f[x_{i+1}, \dots, x_{i+k}] - f[x_i, \dots, x_{i+k-1}]}{x_{i+k} - x_i}$$

Each one is a difference of two shorter divided differences, divided by the spread of the
nodes involved, hence the name. The first one is the familiar slope:

$$f[x_0, x_1] = \frac{f(x_1) - f(x_0)}{x_1 - x_0}$$

## Why they matter

Two reasons, and you will use both this week:

1. **They are the coefficients of Newton's interpolating polynomial** (next notebook).
2. **They estimate derivatives.** There is a mean value theorem for them: for some $\xi$
   in the interval spanned by the nodes,

$$f[x_0, \dots, x_n] = \frac{f^{(n)}(\xi)}{n!}$$

So the first divided difference is a slope, the second is about half the second
derivative, and so on. Divided differences are what a derivative looks like when you only
have samples.

## The table

They are computed in a triangular table, each column from the one before:

| $x_i$ | $f[x_i]$ | 1st | 2nd | 3rd |
|---|---|---|---|---|
| $x_0$ | $f[x_0]$ | $f[x_0,x_1]$ | $f[x_0,x_1,x_2]$ | $f[x_0,x_1,x_2,x_3]$ |
| $x_1$ | $f[x_1]$ | $f[x_1,x_2]$ | $f[x_1,x_2,x_3]$ | |
| $x_2$ | $f[x_2]$ | $f[x_2,x_3]$ | | |
| $x_3$ | $f[x_3]$ | | | |

The **top row** is what we will want later. Store it as a 2D array `T` where `T[i, k]` is
the $k$-th divided difference starting at node $i$; column 0 is just the values, and
column $k$ has $n+1-k$ valid entries.

In [ ]:
# Step 1.1: First divided differences, by hand.
x = np.array([1.0, 2.0, 4.0, 7.0])
y = np.array([1.0, 4.0, 16.0, 49.0])      # this is f(x) = x**2
# Compute the three FIRST divided differences (y[i+1]-y[i])/(x[i+1]-x[i]) and print them.
# Do it in one vectorised line using np.diff, then check one of them by hand in a comment.

In [ ]:
# Step 1.2: The full table.
# Write  divided_difference_table(x, y)  that returns an (n+1) x (n+1) numpy array T
# where T[i, k] is the k-th divided difference starting at node i.
#   - column 0 is y
#   - column k, for i in range(n + 1 - k):
#         T[i, k] = (T[i+1, k-1] - T[i, k-1]) / (x[i+k] - x[i])
#   - leave the unused lower-right entries as np.nan so mistakes are visible
# Hint: start with  T = np.full((n+1, n+1), np.nan)  and use two nested loops.

In [ ]:
# Step 1.3: Test it on something you can verify.
# Run your table on the x, y from step 1.1 (f(x) = x**2).
# For a degree-2 polynomial the 2nd divided difference must equal the leading
# coefficient (1.0) everywhere it is defined, and the 3rd must be 0.
# Print the table and check this. If it fails, fix the function before going on.

In [ ]:
# Step 1.4: A harder test.
# Let  f(x) = x**3 - 2*x + 1  on the nodes [-1, 0, 1, 2, 3].
# Build the table and confirm:
#   - the 3rd divided difference is 1.0 (the leading coefficient)
#   - the 4th divided difference is 0 (to within floating-point noise)
# Use np.isclose for the comparison rather than == . Why does that matter here?

In [ ]:
# Step 1.5: Extract the Newton coefficients.
# Write  newton_coefficients(x, y)  that returns the TOP ROW of the table,
# i.e. the array [f[x0], f[x0,x1], f[x0,x1,x2], ...] of length n+1.
# Test it on the cubic above: you should get the coefficients of the polynomial
# written in Newton form. You will reuse this function in the interpolation notebook.

In [ ]:
# Step 1.6: Divided differences are symmetric.
# f[x0,...,xn] does not depend on the ORDER of the nodes.
# Shuffle the nodes from step 1.4 (keeping each y with its x!), rebuild the table,
# and confirm the HIGHEST divided difference is unchanged, even though the intermediate
# coefficients differ. Hint: build a permutation with rng.permutation(len(x)) and index
# both arrays with it.

In [ ]:
# Step 1.7: The link to derivatives.
# The theorem says   f[x0,...,xn] = f^(n)(xi) / n!   for some xi between the nodes.
# Take f = np.sin and n+1 = 3 nodes clustered tightly around 1.0, say
# [1 - h, 1, 1 + h] for h = 0.01.
# Compute the 2nd divided difference and compare it with  f''(1) / 2! = -sin(1) / 2 .
# Then repeat for h = 0.1 and h = 0.001 and print the errors. What happens as h shrinks?

---
# Exercise 2: Numerical integration (quadrature)

## The idea

We want $I = \int_a^b f(x)\,dx$ but have no antiderivative, or only have $f$ as data.
Every method here does the same thing: **replace $f$ by something you can integrate
exactly** (a constant, a line, a parabola) on each of $n$ small subintervals, and add up
the pieces. With $h = (b-a)/n$ and $x_i = a + ih$:

**Midpoint rule**: approximate $f$ by a constant, sampled at the middle:
$$I \approx h \sum_{i=0}^{n-1} f\!\left(\tfrac{x_i + x_{i+1}}{2}\right)$$

**Trapezoid rule**: approximate $f$ by a straight line on each panel:
$$I \approx h\left[\tfrac{1}{2}f(x_0) + f(x_1) + \dots + f(x_{n-1}) + \tfrac{1}{2}f(x_n)\right]$$

**Simpson's rule**: approximate $f$ by a parabola through each *pair* of panels
(so $n$ must be **even**):
$$I \approx \frac{h}{3}\left[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \dots + 4f(x_{n-1}) + f(x_n)\right]$$

## The error, which is the point

| Rule | Composite error | Exact for |
|---|---|---|
| Midpoint | $O(h^2)$ | degree $\le 1$ |
| Trapezoid | $O(h^2)$ | degree $\le 1$ |
| Simpson | $O(h^4)$ | degree $\le 3$ (!) |

Two things worth noticing. Simpson's rule integrates *cubics* exactly even though it only
fits parabolas: a lucky cancellation that makes it the default choice. And "$O(h^4)$"
means halving $h$ divides the error by **16**: doubling the work buys far more accuracy
than it does for the trapezoid rule.

You will *measure* these exponents in step 2.6, not take them on faith.

In [ ]:
# Step 2.1: Midpoint rule.
# Write  midpoint(f, a, b, n)  returning the composite midpoint approximation.
# Use numpy, not a Python loop: build the midpoints with np.linspace or arange,
# evaluate f on the whole array at once, and sum.

In [ ]:
# Step 2.2: Trapezoid rule.
# Write  trapezoid(f, a, b, n) .
# Hint: with  x = np.linspace(a, b, n + 1)  and  y = f(x) , the sum is
#   h * (y[0]/2 + y[1:-1].sum() + y[-1]/2)

In [ ]:
# Step 2.3: Test on something exact.
# Both rules integrate straight lines exactly. Check that on  f(x) = 3*x + 1  over [0, 2]
# (the true value is 8) both give the right answer even with n = 1 and n = 2.
# Use np.isclose. If this fails, the bug is in your formula, not in the maths.

In [ ]:
# Step 2.4: Test on something curved.
# Integrate  f(x) = x**2  over [0, 1] (true value 1/3) and  f = np.sin  over [0, pi]
# (true value 2) with n = 10, 100, 1000.
# Print the error of each rule at each n. Does the error fall by ~4x when n grows 10x?
# (It should not; think about what O(h^2) predicts for a 10x change, and check that.)

In [ ]:
# Step 2.5: Simpson's rule.
# Write  simpson(f, a, b, n)  and raise a ValueError if n is odd.
# Hint: with y = f(np.linspace(a, b, n+1)), the weights are 1, 4, 2, 4, ..., 4, 1.
# You can build them with  w = np.ones(n+1); w[1:-1:2] = 4; w[2:-1:2] = 2
# Test it on x**2 over [0,1] and on x**3 over [0,1] (true value 1/4): Simpson should
# be exact for BOTH, to machine precision, even with n = 2.

In [ ]:
# Step 2.6: Measure the convergence order.
# For f = np.sin on [0, pi] (true value 2), loop over n = 4, 8, 16, ..., 1024.
# Record the absolute error of all three rules and plot them against h = pi/n
# on a log-log plot (plt.loglog), with a legend and axis labels.
#
# Then MEASURE each slope with np.polyfit(np.log(h), np.log(err), 1)[0].
# You should get about 2, 2 and 4. That number is the convergence order, and measuring
# it like this is the standard way to check that an implementation is correct.
# Note: Simpson's error will hit machine precision and flatten out: exclude those
# points from the fit, or the slope will be nonsense.

In [ ]:
# Step 2.7: Use it for real.
# The integral   \int_0^1 e^{-x^2} dx   has no elementary antiderivative.
# Its true value is 0.7468241328124271.
# Find the smallest n (try powers of 2) for which Simpson's rule agrees with that
# to within 1e-10, and report the error of the trapezoid rule at the same n.

In [ ]:
# Step 2.8: Integrate DATA, not a function.
# You are given samples, not a formula; which is the normal situation in a lab:
x_data = np.linspace(0, 4, 21)
y_data = np.exp(-x_data) * np.cos(2 * x_data)
# Apply the trapezoid rule directly to these samples (you cannot call f at new points!)
# and compare with np.trapz(y_data, x_data).
# Then compare against the true integral, 0.2077812766 (to 10 dp).

---
# Exercise 3: Numerical differentiation

## The formulas

From Taylor's theorem, with step $h$:

**Forward difference**: error $O(h)$:
$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$

**Central difference**: error $O(h^2)$, and free: it costs the same two evaluations:
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

**Second derivative**: error $O(h^2)$:
$$f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2}$$

The central difference is more accurate because the odd-order terms in the two Taylor
expansions cancel. Always prefer it unless you cannot evaluate $f$ on both sides.

## The catch, which integration does not have

Integration is *averaging*, so errors cancel. Differentiation is *differencing*, so errors
amplify. There are two competing sources of error:

- **Truncation error**: the maths, $O(h)$ or $O(h^2)$. Shrinks as $h \to 0$.
- **Round-off error**: subtracting two nearly equal floats destroys significant digits,
  then you *divide by a tiny $h$*, magnifying what is left. Grows like $O(\varepsilon/h)$.

Total error is roughly the sum, so it is **U-shaped in $h$**: there is a best step size,
and going smaller makes things *worse*. Balancing the two terms gives

$$h_{\text{opt}} \approx \sqrt{\varepsilon} \approx 1.5\times10^{-8} \quad \text{(forward)},
\qquad h_{\text{opt}} \approx \varepsilon^{1/3} \approx 6\times10^{-6} \quad \text{(central)}$$

where $\varepsilon \approx 2.2\times 10^{-16}$ is machine epsilon. This is the single most
important practical fact in this notebook: **$h = 10^{-15}$ is not "more accurate", it is
garbage.** You will plot this curve yourself in step 3.4.

In [ ]:
# Step 3.1: Forward difference.
# Write  forward_diff(f, x, h) .
# Test it on f = np.sin at x = 1.0 with h = 1e-5. The true answer is np.cos(1.0).
# Print the approximation, the truth, and the absolute error.

In [ ]:
# Step 3.2: Central difference.
# Write  central_diff(f, x, h)  and test it at the same x and h.
# Its error should be many orders of magnitude smaller for the same cost. By how much?

In [ ]:
# Step 3.3: Confirm the orders.
# For h = 1e-1, 1e-2, 1e-3, 1e-4, print the error of both rules.
# Each time h shrinks 10x, forward-difference error should fall ~10x and central ~100x.
# Check that in the printed numbers before you trust the plot in the next step.

In [ ]:
# Step 3.4: The U-shaped error curve. (The important one.)
# Take h = np.logspace(-16, -1, 200) and compute the error of BOTH rules at x = 1.0
# for f = np.sin. Plot them with plt.loglog, label the axes, add a legend.
#
# Then find the h that minimises each curve (np.argmin) and print them.
# Compare with the predicted sqrt(eps) and eps**(1/3) from the notes above.
# np.finfo(float).eps gives machine epsilon.
# Write one sentence in a comment about what the left half of the plot means.

In [ ]:
# Step 3.5: Second derivative.
# Write  second_diff(f, x, h)  using the three-point formula.
# Test on f = np.sin at x = 1.0 (true value -sin(1)) with a sensible h.
# Then repeat the U-curve experiment: this formula divides by h**2, so round-off hurts
# even sooner. Find its optimal h and compare with the first-derivative case.

In [ ]:
# Step 3.6: Differentiate an array of samples.
# Again the realistic case: data, no formula.
x_data = np.linspace(0, 2 * np.pi, 50)
y_data = np.sin(x_data)
# Estimate the derivative at every interior point with the central difference applied to
# the ARRAY (hint: (y[2:] - y[:-2]) / (x[2:] - x[:-2]); no loop, no calls to f).
# Plot your estimate against the true cos(x), and plot the error separately.
# Then compare with np.gradient(y_data, x_data), which also handles the endpoints.

In [ ]:
# Step 3.7: Stretch: Richardson extrapolation.
# The central difference has error  D(h) = f'(x) + C*h**2 + O(h**4).
# So the combination   (4*D(h/2) - D(h)) / 3   cancels the h**2 term, leaving O(h**4).
# Implement it, and compare its error against plain central differences at h = 1e-2.
# You should gain several digits for one extra function evaluation.

---
# Exercise 4: Newton's method

## The idea

To solve $f(x) = 0$: stand at $x_k$, replace $f$ by its tangent line there, and jump to
where the *tangent* hits zero. That gives

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$$

Repeat. It is Taylor's theorem truncated after one term, applied over and over.

## Why it is worth the derivative

Near a **simple root**, Newton's method converges **quadratically**: if $e_k = x_k - r$,
then

$$|e_{k+1}| \approx C\,|e_k|^2$$

In plain terms, **the number of correct digits roughly doubles every iteration.** Starting
from 2 correct digits you get 4, then 8, then 16; machine precision in about four steps.
Bisection, by contrast, gains one *bit* per step. That difference is why Newton is
everywhere: root finding, optimisation, and the training of every neural network you will
meet later this week.

## When it fails

It is fast, not safe. It fails when:

- $f'(x_k) = 0$: you divide by zero, or nearly so, and get flung across the number line
- the initial guess is too far away: it can diverge or converge to a *different* root
- the iteration **cycles**: e.g. $f(x) = x^3 - 2x + 2$ from $x_0 = 0$ bounces $0 \to 1
  \to 0 \to 1 \dots$ forever
- the root is **repeated** (e.g. $f = (x-1)^2$), where $f'$ vanishes at the root too and
  convergence degrades from quadratic to merely linear

So a real implementation always carries a maximum iteration count, and production code
usually pairs Newton with a safe fallback like bisection.

## Stopping

Stop when $|f(x_k)|$ is small, *and/or* when $|x_{k+1} - x_k|$ is small. Neither test
alone is trustworthy: the first can be fooled by a very flat $f$, the second by very slow
progress. And never test against $0$: always against a tolerance.

In [ ]:
# Step 4.1: One step, by hand.
# Solve x**2 - 2 = 0 (the root is sqrt(2)).
# Define f and its derivative fprime, start at x = 1.0, and compute ONE Newton step
# by writing the formula out explicitly. Print the new x and its error against
# np.sqrt(2).

In [ ]:
# Step 4.2: Five steps in a loop.
# Repeat the step above 5 times in a for loop, printing a table each iteration:
# the iteration number, x, f(x), and the absolute error.
# Watch the error column: count how many correct digits you gain per line.

In [ ]:
# Step 4.3: Package it properly.
# Write  newton(f, fprime, x0, tol=1e-12, max_iter=50)  that returns
# (root, history) where history is the list of iterates, and that:
#   - stops when abs(f(x)) < tol
#   - raises (or returns a clear failure) if abs(fprime(x)) is smaller than, say, 1e-14
#   - stops after max_iter iterations rather than looping forever
# Test it on x**2 - 2 from x0 = 1.0 and check the result against np.sqrt(2).

In [ ]:
# Step 4.4: Verify quadratic convergence numerically.
# Using the history from 4.3, compute the errors e_k = |x_k - sqrt(2)| and print the
# ratio   e_{k+1} / e_k**2   for each step.
# If convergence is quadratic this ratio settles down to a roughly constant value.
# For comparison, print e_{k+1} / e_k as well; that ratio should be heading to 0.
# (Drop the last iterates once the error is at machine precision; the ratios there
# are meaningless noise.)

In [ ]:
# Step 4.5: A repeated root.
# Run your solver on  f(x) = (x - 1)**2 , which has a double root at x = 1.
# Print the error each iteration and compute e_{k+1}/e_k .
# You should find that ratio settling near 0.5: linear convergence, one bit a step,
# not quadratic. Explain in a comment what goes wrong, in terms of f'(root).

In [ ]:
# Step 4.6: Make it fail.
# Run your solver on  f(x) = x**3 - 2*x + 2  starting from x0 = 0.0 .
# Print the iterates. You should see it cycle between 0 and 1 forever, and your
# max_iter guard should stop it. Verify the cycle by computing one step from 0 by hand
# in a comment.
# Then find a starting point from which it DOES converge, and report the root.

In [ ]:
# Step 4.7: Newton without a derivative (secant method).
# If f' is unavailable, replace it with the divided difference from Exercise 1:
#     x_{k+1} = x_k - f(x_k) * (x_k - x_{k-1}) / (f(x_k) - f(x_{k-1}))
# Implement  secant(f, x0, x1, tol=1e-12, max_iter=50)  and solve x**2 - 2 again.
# Count the iterations it needs versus Newton. Its order of convergence is about 1.618,
# so it needs a few more steps, but no derivative at all.

In [ ]:
# Step 4.8: Put the whole notebook together.
# Solve   cos(x) = x   (the root is about 0.739085).
# Do it THREE ways and compare the iteration counts:
#   (a) Newton, with the derivative worked out by hand
#   (b) Newton, with the derivative supplied by your central_diff from Exercise 3
#       (pick h sensibly; you now know exactly which h to use)
#   (c) the secant method
# Does the numerical derivative cost you any accuracy in the final root? Should it?

---
## What to take away

- **Every approximation has an order.** Measuring the slope on a log-log plot is how you
  confirm your code matches the theory; and it is the fastest way to catch a bug that
  leaves the answer plausible but wrong.
- **Smaller $h$ is not always better.** Integration rewards refinement; differentiation
  punishes it past a point, because of round-off. Know which regime you are in.
- **Test against cases you can verify exactly.** A polynomial of low degree, a straight
  line, a known integral. If a method is exact for those and it is not exact for yours,
  the bug is in your code, not in the mathematics.
- **Fast methods are usually fragile.** Newton's quadratic convergence comes with real
  failure modes, so guard every loop with a maximum iteration count.

Next: the **interpolation** notebook, which picks up divided differences from Exercise 1
and turns them into polynomials.